In [1]:
import json 
import spacy
import scispacy
from scispacy.linking import EntityLinker
import pubmed_parser as pp
from tqdm import tqdm

nlp = spacy.load("en_core_sci_lg")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

nlp2 = spacy.load('en_ner_bc5cdr_md')
nlp2.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

dicts_out = pp.parse_medline_xml('../data/pubmed_data/processed_data/processed_full_data.xml',
                                 year_info_only=False,
                                 nlm_category=False,
                                 author_list=False,
                                 reference_list=False)

/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/spacy/util.py:833: UserWarning: [W095] Model 'en_core_sci_lg' (0.4.0) was trained with spaCy v3.0 and may not be 100% compatible with the current version (3.2.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfTransformer from version 0.20.3 when using version 1.0.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/rgoli/.local/lib/python3.8/site-packages/sklearn/base.py:324: UserWarning: Trying to unpickle estimator TfidfVectorizer from version 0.20.3 w

In [2]:
## HoldOut 125 GS + 111 No/Little Abstract

pmids_HO = set()
with open('../data/HoldOut_Total.txt','r') as f:
    for line in f.readlines():
        pmids_HO.add(line.strip())
print("Hold Out Data: ",len(pmids_HO))

Hold Out Data:  236


In [3]:
N=len(dicts_out)
i=0
j=0
while i<N:
    if dicts_out[i]['pmid'] not in pmids_HO:
        dicts_out[j]=dicts_out[i]
        j+=1
    i+=1
dicts_out[j:]=[]

print("Full Data for further processing: ",len(dicts_out))

Full Data for further processing:  3100


In [4]:
comb_arr=[]
pmid_arr =[]
for paper in dicts_out:
    comb_arr.append(paper['title']+' '+paper['abstract'])
    pmid_arr.append(paper['pmid'])
    
idx=0
processed_texts = {}
scispacy_kw = {}
no_kws_trimmed =[]
no_kws=[]
post_no_kws =[]
outlier_kws=[
    'http','www','university','department','antibiotic','antimicrobial','institute','ministry', 'pubmed',
    '.gov','.org','.com','.edu','.net',
    'city','disease','injury','trauma','syndrome','country','national','regimen','swelling','cholesterol','cerebrovascular','leukemia'
    'surgery', 'medication','infection','stroke','diabetes','bleeding','comorbid','java','python',
    "united states",'united kingdom','india','china','germany','france','ghana','australia','italy','england','japan',
    'english','spanish','french','british','spain'
    ]

remove_sent_no_outliers = []
with tqdm(nlp.pipe(comb_arr),total=len(comb_arr)) as pbar:
    for doc in pbar:
        sents = [[token.orth_.lower() for token in sent if not token.is_punct | token.is_space] for sent in doc.sents]
        if len(sents)<4:
            remove_sent_no_outliers.append(idx)
            idx+=1
            continue
        text = [token.orth_ for token in doc if not token.is_punct | token.is_space] 
        rule_out_ents = [str(_) for _ in nlp2(comb_arr[idx]).ents]
        rule_out_ents = [_ for _ in rule_out_ents if not _.isupper()]
        kws = [str(_) for _ in doc.ents]
        # print('-'*50)
        # print(len(kws))
        temp_kws_len = len(kws)
        # print(kws)
        if rule_out_ents==[]:
            kws = [_ for _ in kws]
        else:
            kws = [_ for _ in kws if _ not in rule_out_ents]
        # print(kws)
        kws = [_ for _ in kws if all([True if n_ not in _.lower() else False for n_ in outlier_kws])]
        no_kws_trimmed.append(temp_kws_len-len(kws))
        no_kws.append(temp_kws_len)
        post_no_kws.append(len(kws))
        # print(len(kws))
        # print(rule_out_ents)
        # print(kws)
        # print('-'*50)
        kws.sort(reverse=True, key=len)
        
        dicts_out[idx]['fullText'] = text
        dicts_out[idx]['sentences'] = sents
        dicts_out[idx]["keywords"]=kws=kws
        
        idx+=1
        pbar.set_description("The %s document is processed" % (idx + 1))

## Filter Docs with less than minimum senetences
i=0
j=0
print("Total Articles removed: ",len(remove_sent_no_outliers))
print([pmid_arr[x] for x in remove_sent_no_outliers])
while i<idx:
    if i not in remove_sent_no_outliers:
        dicts_out[j]=dicts_out[i]
        j+=1
    i+=1
dicts_out[j:]=[]

dataset_length = len(dicts_out)

print("Total Dataset Length: ",dataset_length)
print("Train Dataset Length: ",dataset_length//3)
print("Test  Dataset Length: ",dataset_length-dataset_length//3)

with open("GNA_Files/train.json", "w") as outfile:
    json.dump(dicts_out[:dataset_length//3], outfile)

with open("GNA_Files/test.json", "w") as outfile:
    json.dump(dicts_out[dataset_length//3:], outfile)

print("Total KWS generated:", sum(no_kws))
print("Avg. KWS generated/abstract:", sum(no_kws)/len(no_kws))

print("Total KWS trimmed: ",sum(no_kws_trimmed))
print("Avg. KWS trimmed:", sum(no_kws_trimmed)/len(no_kws_trimmed))

print("Total KWS - post processing: ",sum(post_no_kws))
print("Avg. KWS - post processing:", sum(post_no_kws)/len(post_no_kws))

  0%|          | 0/3100 [00:00<?, ?it/s]/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/scispacy/candidate_generation.py:284: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  extended_neighbors[empty_vectors_boolean_flags] = numpy.array(neighbors)[:-1]
/home/rgoli/.conda/envs/pytorch/lib/python3.8/site-packages/scispacy/candidate_generation.py:285: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  extended_distances[empty_vectors_boolean_flags] = numpy.array(distances)[:-1]
The 3101 document is processed: 100%|██████████| 3100/3100 [04:37<00:00,

Total Articles removed:  48
['34042851', '31349252', '31225983', '30399617', '28269917', '28086985', '27762202', '27235214', '26262257', '26262255', '23920886', '23920721', '23861123', '23851288', '22929351', '22219234', '22060071', '21295751', '21095686', '20592007', '20501532', '20407168', '18999221', '18999012', '18998900', '18998875', '18694272', '18694237', '18580428', '18487843', '17945725', '17911954', '17238710', '17238621', '17238566', '17238503', '16779188', '15846970', '15770911', '15455842', '14728543', '12365297', '11813521', '10587916', '10338700', '9700425', '10177918', '10177693']
Total Dataset Length:  3052
Train Dataset Length:  1017
Test  Dataset Length:  2035
Total KWS generated: 224833
Avg. KWS generated/abstract: 73.66743119266054
Total KWS trimmed:  11995
Avg. KWS trimmed: 3.9302096985583224
Total KWS - post processing:  212838
Avg. KWS - post processing: 69.73722149410223


In [5]:
train_data = dicts_out[:dataset_length//3]
test_data = dicts_out[dataset_length//3:]

In [6]:
train_corpus_sentences = []
for x in train_data:
    train_corpus_sentences.extend(x['sentences'])
print("Total Sentences in Train Corpus: ", len(train_corpus_sentences))

test_corpus_sentences = []
for x in test_data:
    test_corpus_sentences.extend(x['sentences'])
print("Total Sentences in Test Corpus: ", len(test_corpus_sentences))

Total Sentences in Train Corpus:  12023
Total Sentences in Test Corpus:  20555


In [7]:
print('Sample: ', train_corpus_sentences[0])

Sample:  ['continuous', 'cardiorespiratory', 'monitoring', 'is', 'a', 'dominant', 'source', 'of', 'predictive', 'signal', 'in', 'machine', 'learning', 'for', 'risk', 'stratification', 'and', 'clinical', 'decision', 'support']


In [8]:
import gensim

modelW2V = gensim.models.Word2Vec(sentences=train_corpus_sentences,size=300,window=5,min_count=0,workers=10,sg=1,seed=42,compute_loss=True)

In [10]:
# modelW2V.save('GNA_Files/word2vec.embedding')
modelW2V.wv.save_word2vec_format('GNA_Files/word2vec.bin', binary=True)

In [11]:
# modelW2V = gensim.models.Word2Vec.load('GNA_Files/word2vec.bin')
from gensim.models import KeyedVectors

filename = 'GNA_Files/word2vec.bin'
model = KeyedVectors.load_word2vec_format(filename, binary=True)

In [13]:
model.vectors

array([[ 0.01567691, -0.01434722, -0.05928158, ...,  0.27488738,
         0.11898029,  0.1458889 ],
       [-0.26982486,  0.08258816,  0.08433735, ...,  0.2349544 ,
         0.14525235,  0.202715  ],
       [-0.2517215 ,  0.09499723,  0.01313456, ...,  0.286812  ,
         0.2631154 ,  0.03964536],
       ...,
       [-0.03499442,  0.00231899,  0.018925  , ...,  0.0207335 ,
         0.0037053 ,  0.0063251 ],
       [-0.0469954 ,  0.00591397,  0.03081469, ...,  0.05419372,
         0.02340999,  0.01806109],
       [-0.04363944,  0.00065229,  0.02873107, ...,  0.03676672,
         0.02711157,  0.01157401]], dtype=float32)

In [12]:
modelW2V.wv.most_similar(positive=['clinical','decision'], topn = 10)

[('computerized', 0.9305931925773621),
 ('computer', 0.9166833758354187),
 ('computerised', 0.9118633270263672),
 ('web-based', 0.9083110094070435),
 ('cpoe', 0.9054160118103027),
 ('guideline-based', 0.9039593935012817),
 ('decision-support', 0.9011868238449097),
 ('point-of-care', 0.8996912240982056),
 ('knowledge-based', 0.8965176343917847),
 ('phone-based', 0.8961668014526367)]

In [11]:
modelW2V.wv.most_similar(positive=['cardiorespiratory'], topn = 10)

[('atc', 0.9970329403877258),
 ('speech', 0.9967970252037048),
 ('connection', 0.9967836737632751),
 ('un', 0.9967827200889587),
 ('dg', 0.9967769384384155),
 ('javascript', 0.9967178106307983),
 ('salud', 0.9966831207275391),
 ('camera', 0.9966326951980591),
 ('preferably', 0.9965840578079224),
 ('atypical', 0.9965786933898926)]

In [12]:
modelW2V.wv.most_similar(positive=['machine','learning'], topn = 10)

[('networks', 0.9443327188491821),
 ('fuzzy', 0.9412860870361328),
 ('artificial', 0.9396160244941711),
 ('techniques', 0.9369198679924011),
 ('deep', 0.9350016713142395),
 ('ml', 0.9307934641838074),
 ('neural', 0.9299061298370361),
 ('algorithms', 0.9290131330490112),
 ('bayesian', 0.925586462020874),
 ('intelligence', 0.9251716136932373)]

In [13]:
modelW2V.wv.most_similar(positive=['breakfast'], topn = 10)

KeyError: "word 'breakfast' not in vocabulary"

In [ ]:
## Word2Vec 300 Dim vector for a word
# model.wv['machine']

In [14]:
from gensim.models.fasttext import FastText as FT_gensim
modelFT = FT_gensim(size=300)

# build the vocabulary
modelFT.build_vocab(sentences=train_corpus_sentences)
# help(model.build_vocab)

# train the model
modelFT.train(
    sentences=train_corpus_sentences, epochs=modelFT.epochs,
    total_examples=modelFT.corpus_count, total_words=modelFT.corpus_total_words
)

In [19]:
modelFT.wv.save_word2vec_format('GNA_Files/fastText.bin', binary=True)

In [16]:
modelFT.save('GNA_Files/fastText.embedding')
modelFT = FT_gensim.load('GNA_Files/fastText.embedding')

In [18]:
from pprint import pprint
pprint('breakfast' in modelFT.wv.vocab)

False


In [19]:
# model['breakfast']
print(modelFT.most_similar("breakfast"))

/local_scratch/pbs.5214499.pbs02/ipykernel_3872339/923396621.py:2: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  print(modelFT.most_similar("breakfast"))


[('neuromuscular', 0.9990766644477844), ('venous', 0.9988112449645996), ('screens', 0.9986350536346436), ('scholar', 0.9982324838638306), ('drain', 0.9980489015579224), ('discover', 0.9979493618011475), ('psychiatrists', 0.9978885650634766), ('survivors', 0.9978170990943909), ('coverage', 0.9977561235427856), ('calculator', 0.9977135062217712)]


In [20]:
modelFT.wv.most_similar(positive=['machine','learning'], topn = 10)

[('machine-learning', 0.9995595216751099),
 ('rule', 0.9936349391937256),
 ('systematically', 0.991623044013977),
 ('machines', 0.990259051322937),
 ('form', 0.9896106719970703),
 ('designing', 0.9893839955329895),
 ('ml-cdss', 0.9885512590408325),
 ('cdst', 0.9884353876113892),
 ('run', 0.9883862733840942),
 ('e-cdss', 0.9881868362426758)]

In [21]:
modelFT.wv.most_similar(positive=['cardiorespiratory'], topn = 10)

[('disparities', 0.9978469610214233),
 ('acute', 0.9976223707199097),
 ('comorbidities', 0.9975756406784058),
 ('blunt', 0.9974039196968079),
 ('syndromes', 0.9968019127845764),
 ('diverse', 0.9965807199478149),
 ('anxiety', 0.9961704015731812),
 ('measuring', 0.9959949254989624),
 ('advocacy', 0.9959644675254822),
 ('diastolic', 0.9957002997398376)]

In [22]:
modelFT.wv.most_similar(positive=['clinical','decision'], topn = 10)

[('decision-support', 0.9969140291213989),
 ('decision-making', 0.9876968264579773),
 ('compute', 0.9854162335395813),
 ('decisional', 0.9854007959365845),
 ('systems', 0.9798752069473267),
 ('decisions', 0.9790632128715515),
 ('tools', 0.9790303111076355),
 ('ecosystem', 0.978584885597229),
 ('computing', 0.977027177810669),
 ('tool', 0.9766429662704468)]

In [23]:
modelFT.wv.most_similar(positive=['CDSS'], topn = 10)

[('fn', 0.09876846522092819),
 ('patients', 0.07864515483379364),
 ('inpatients', 0.06329755485057831),
 ('outpatients', 0.06289007514715195),
 ('events', 0.05562923103570938),
 ('with', 0.05228497087955475),
 ('at', 0.05152835696935654),
 ('primary', 0.04952406510710716),
 ('patient', 0.04809639975428581),
 ('agents', 0.0472579225897789)]

In [24]:
modelFT.wv.most_similar(positive=['CDS'], topn = 10)

[('primary', 0.1063309758901596),
 ('care', 0.08499588817358017),
 ('patients', 0.07769837230443954),
 ('with', 0.07206980884075165),
 ('fn', 0.07032357156276703),
 ('patient', 0.07031993567943573),
 ('in', 0.06650303304195404),
 ('inpatients', 0.0640908032655716),
 ('primarily', 0.05997277796268463),
 ('outpatients', 0.055154383182525635)]

In [25]:
modelFT.wv.most_similar(positive=['clinical','CDS'], topn = 10)

[('integral', 0.6920782327651978),
 ('making', 0.6916896104812622),
 ('into', 0.6899747848510742),
 ('integrate', 0.6885728240013123),
 ('clinic', 0.687396228313446),
 ('integrates', 0.6854591965675354),
 ('supporting', 0.6835291981697083),
 ('providing', 0.6830019354820251),
 ('intelligent', 0.6807341575622559),
 ('inform', 0.6798173189163208)]

In [26]:
# modelW2V.wv['machine']

In [27]:
# modelFT.wv['machine']

In [28]:
modelW2V.wv.most_similar(positive=['machine'])

[('learning', 0.9689337015151978),
 ('fuzzy', 0.9415723085403442),
 ('networks', 0.941362202167511),
 ('artificial', 0.9370601773262024),
 ('deep', 0.9325191974639893),
 ('techniques', 0.9301704168319702),
 ('neural', 0.9299391508102417),
 ('algorithm', 0.9274633526802063),
 ('ml', 0.9274008274078369),
 ('vector', 0.9235712885856628)]

In [29]:
modelFT.wv.most_similar(positive=['machine'])

[('framework', 0.9927793145179749),
 ('a', 0.9883780479431152),
 ('run', 0.987257182598114),
 ('app', 0.986479640007019),
 ('desiree', 0.9864590167999268),
 ('ai', 0.9862563610076904),
 ('desire', 0.9848883152008057),
 ('work', 0.9838255643844604),
 ('designs', 0.9830452799797058),
 ('developers', 0.9828075170516968)]

In [30]:
print(modelFT.wv.similarity('CDSS', 'clinical'))
print(modelFT.wv.similarity('CDSS', 'decision'))
print(modelFT.wv.similarity('CDSS', 'support'))
print(modelFT.wv.similarity('CDSS', 'systems'))

-0.061783798
-0.05707564
-0.07312648
-0.06821948


In [31]:
print(modelW2V.wv.similarity('CDSS', 'clinical'))
print(modelW2V.wv.similarity('CDSS', 'decision'))
print(modelW2V.wv.similarity('CDSS', 'support'))
print(modelW2V.wv.similarity('CDSS', 'systems'))

KeyError: "word 'CDSS' not in vocabulary"